In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np

# all_classes = np.array(['c', 'cpp', 'js', 'py'])

In [ ]:
def get_windows(content: str, window_size=30, stride=15):
    lines = content.splitlines()

    snippets = []
    for i in range(0, len(lines) - window_size + 1, stride):
        window = lines[i:i + window_size]

        if len("".join(window).strip()) > 50:
            snippets.append("\n".join(window))
    return snippets

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix, hstack

class MetaFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer
        self.scaler = StandardScaler(with_mean=False)
        
    def fit(self, X, y=None):
        X_meta = self.extract_meta_features(pd.Series(X))
        self.scaler.fit(X_meta.values)
        return self

    def extract_meta_features(self, text_series) -> pd.DataFrame:
        features = pd.DataFrame()
        features['semicolon_density'] = text_series.str.count(';') / text_series.str.len()
        features['brace_ratio'] = (text_series.str.count('{') + text_series.str.count('}')) / text_series.str.len()
        features['pointer_marker'] = text_series.str.contains(r'\w+\*').astype(int)
        features['python_marker'] = text_series.str.contains(r'\bdef\b|\belif\b').astype(int)
        features['cpp_marker'] = text_series.str.contains(r'std::|template<|public:|private:').astype(int)
        features['js_marker'] = text_series.str.contains(r'\blet\b|\bfunction\b|=>').astype(int)


        # C++
        features['has_class'] = text_series.str.contains(r'\bclass\b').astype(int)
        features['has_namespace'] = text_series.str.contains(r'\bnamespace\b').astype(int)
        features['has_template'] = text_series.str.contains(r'\btemplate\b').astype(int)
        features['has_using'] = text_series.str.contains(r'\busing\b').astype(int)
        features['has_cpp_keywords'] = text_series.str.contains(
            r'\bnew\b|\bdelete\b|\bcout\b|\bcin\b|\bpublic:\b|\bprivate:\b|\bprotected:\b|\bvirtual\b|\boverride\b|\bconstexpr\b|\bauto\b|\bnullptr\b'
        ).astype(int)
        features['scope_resolution'] = text_series.str.count('::') / (text_series.str.len() + 1)
        
        # C specific (but not in C++)
        features['has_c_keywords'] = text_series.str.contains(r'\bprintf\b|\bscanf\b|\bmalloc\b|\bfree\b').astype(int)
        
        # Python specific
        features['has_python_decorator'] = text_series.str.contains(r'@\w+').astype(int)
        features['has_self'] = text_series.str.contains(r'\bself\b').astype(int)
        features['has_import_from'] = text_series.str.contains(r'\bfrom\s+\S+\s+import\b').astype(int)
        
        # JavaScript specific
        features['has_arrow_func'] = text_series.str.contains(r'=>').astype(int)
        features['has_let_const'] = text_series.str.contains(r'\blet\b|\bconst\b').astype(int)
        features['has_console'] = text_series.str.contains(r'\bconsole\.').astype(int)
        
        # Preprocessor directives (C/C++)
        features['preprocessor_count'] = text_series.str.count(r'^\s*#')
        
        # Comment styles
        features['cpp_comment_density'] = text_series.str.count(r'//') / (text_series.str.len() + 1)
        features['c_comment_density'] = text_series.str.count(r'/\*.*?\*/') / (text_series.str.len() + 1)
        features['python_comment_density'] = text_series.str.count(r'#') / (text_series.str.len() + 1)
        return features.fillna(0)
    
    def transform(self, X):
        X_text = self.vectorizer.transform(X)

        X_meta = self.extract_meta_features(pd.Series(X))
        X_meta_scaled = self.scaler.transform(X_meta.values)
        
        return hstack([X_text, csr_matrix(X_meta_scaled)])

In [ ]:
df_iter = pd.read_parquet(
    'datasets/dataset-v2.parquet',
    columns=['Content', 'Language']
).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
all_classes = df_iter['Language'].values.unique().to_numpy()
all_labels = df_iter['Language'].values

weights = compute_class_weight(class_weight='balanced', classes=all_classes, y=all_labels)
weight_dict = dict(zip(all_classes, weights))

In [ ]:
HASH_FEATURES = 2 ** 20

vectorizer = HashingVectorizer(
    n_features=HASH_FEATURES,
    analyzer='word',
    ngram_range=(1, 4),
    alternate_sign=False,
    lowercase=False
)

model = SGDClassifier(
    loss='hinge',
    penalty='l2',
    eta0=0.01,
    learning_rate='adaptive',
    alpha=1e-5,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42,
    class_weight=weight_dict
    # early_stopping=True
)

pipeline = make_pipeline(
    MetaFeatureExtractor(vectorizer),
    model
)

In [ ]:
chunksize = 50_000

import random
random.seed(42)
test_size = 0.2

X_train, y_train = [], []
X_test, y_test = [], []

In [ ]:
sample_size = 1000
sample_texts = []
sample_langs = []

for i, row in df_iter.iterrows():
    if len(sample_texts) >= sample_size:
        break
    windows = get_windows(row['Content'])
    if windows:
        sample_texts.append(windows[0])
        sample_langs.append(row['Language'])

pipeline[0].fit(sample_texts, sample_langs)

In [ ]:
train_size = 0

random.seed(42)

for i, row in df_iter.iterrows():
    content = row['Content']
    lang = row['Language']

    windows = get_windows(content)
    
    if random.random() > test_size:
        for s in windows:
            X_train.append(s)
            y_train.append(lang)

            if len(X_train) >= chunksize:
                X_batch = pipeline[0].transform((X_train))
                pipeline[1].partial_fit(X_batch, y_train, classes=all_classes)

                train_size += len(y_train)
                X_train.clear()
                y_train.clear()
    else:
        X_test.extend(windows)
        y_test.extend([lang] * len(windows))
    
    if i % 10_000 == 0:
        print(f"Train step = {i}")

if X_train:
    X_batch = pipeline[0].transform((X_train))
    pipeline[1].partial_fit(X_batch, y_train, classes=all_classes)
    train_size += len(y_train)
    X_train.clear()
    y_train.clear()

In [ ]:
import joblib

joblib.dump(pipeline, 'models/sgdc-pipeline-5.joblib', compress=True)

In [ ]:
# min_count, train_size, len(X_test), snippet_counts
train_size, len(X_test)

# (322323, 73472)  sgdc-pipeline-4
# (803995, 177866) sgdc-pipeline-5

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

preds = pipeline.predict(X_test)
loss = accuracy_score(y_test, preds)
print(loss)

# sgdc-2 0.8702186491883865
# sgdc-3 0.9263218722912453
# sgdc-pipeline-1 0.5814813866557428
# sgdc-pipeline-2 0.8971091511087615
# sgdc-pipeline-3 0.907978549651568
# sgdc-pipeline-4 0.9276595165505227
# sgdc-pipeline-5 0.9419844152339402

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


cm = confusion_matrix(y_test, preds)
print(cm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap='Blues', values_format='d')
plt.show()


In [ ]:
import joblib

pipeline = joblib.load('models/sgdc-pipeline-5.joblib')

In [ ]:
import pandas as pd

my_test_file = ""
with open('test_file.txt', 'r') as file:
    my_test_file = file.read()

print(my_test_file)
pipeline.predict([my_test_file])